# 02 — Exploratory Data Analysis

Requires `data/processed/merged_panel.parquet`, `panel_logistic_2021_2025.parquet`,
and `panel_cph_2018_2025.parquet` from notebook 01.

Covers spec §10 Phase 3 charts:
1. Distributions of FICO, LTV, DTI by origination cohort
2. Default rate by FICO bucket, LTV bucket, property state
3. FICO vs. Default boxplot
4. Delinquency status distribution
5. Loan age distribution (right-censoring check)
6. Zero balance code breakdown
7. Vintage default rate chart (by origination quarter)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path('..') / 'src'))

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED = Path('..') / 'data' / 'processed'
FIGURES   = Path('..') / 'artifacts' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the two model-specific panels (never cross-load per spec §4.6)
panel = pd.read_parquet(PROCESSED / 'panel_logistic_2021_2025.parquet')
panel_cph = pd.read_parquet(PROCESSED / 'panel_cph_2018_2025.parquet')

# Derive origination year/quarter from first_payment_date
panel['orig_dt'] = pd.to_datetime(
    panel['first_payment_date'].astype(str), format='%Y%m', errors='coerce'
)
panel['orig_year'] = panel['orig_dt'].dt.year
panel['orig_quarter'] = panel['orig_dt'].dt.to_period('Q')

# Strict default flag (spec §4.3)
terminal_mask = panel['zero_balance_code'].isin(['03', '06', '09'])
dpd90_mask    = pd.to_numeric(panel['delinquency_status'], errors='coerce').fillna(0) >= 3
panel['default'] = (terminal_mask | dpd90_mask).astype(int)

# Loan-level collapse for classification analyses
loan_level = panel.groupby('loan_id').agg(
    credit_score   = ('credit_score',   'first'),
    ltv            = ('ltv',            'first'),
    dti            = ('dti',            'first'),
    orig_loan_term = ('orig_loan_term', 'first'),
    property_state = ('property_state', 'first'),
    amortization_type = ('amortization_type', 'first'),
    occupancy_status  = ('occupancy_status',  'first'),
    orig_year      = ('orig_year',      'first'),
    orig_quarter   = ('orig_quarter',   'first'),
    default        = ('default',        'max'),
    max_loan_age   = ('loan_age',       'max'),
).reset_index()

print(f'Panel (2021-2025): {panel.shape[0]:,} performance rows, {loan_level.shape[0]:,} unique loans')
print(f'Loan-level default rate: {loan_level.default.mean():.4%}')

## 1. Distributions of FICO, LTV, DTI by origination cohort

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for year, grp in loan_level.groupby('orig_year'):
    axes[0].hist(grp['credit_score'].dropna(), bins=40, alpha=0.5, label=str(year), density=True)
    axes[1].hist(grp['ltv'].dropna().clip(0, 105), bins=40, alpha=0.5, density=True)
    axes[2].hist(grp['dti'].dropna().clip(0, 65), bins=40, alpha=0.5, density=True)

axes[0].set_xlabel('FICO Score'); axes[0].set_title('FICO by Cohort'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('LTV (%)');    axes[1].set_title('LTV by Cohort')
axes[2].set_xlabel('DTI (%)');    axes[2].set_title('DTI by Cohort')

plt.tight_layout()
plt.savefig(FIGURES / 'eda_fico_ltv_dti_distributions.png', bbox_inches='tight')
plt.show()

## 2. Default rate by FICO bucket, LTV bucket, property state

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# FICO buckets
fico_bins = [300, 580, 620, 660, 700, 740, 780, 850]
fico_labels = ['<580', '580-619', '620-659', '660-699', '700-739', '740-779', '780+']
loan_level['fico_bucket'] = pd.cut(
    loan_level['credit_score'], bins=fico_bins, labels=fico_labels, right=False
)
fico_dr = loan_level.groupby('fico_bucket', observed=True)['default'].mean() * 100
fico_dr.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Default Rate by FICO Bucket')
axes[0].set_ylabel('Default Rate (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=2))

# LTV buckets
ltv_bins = [0, 60, 70, 80, 90, 97, 105]
ltv_labels = ['<60', '60-70', '70-80', '80-90', '90-97', '>97']
loan_level['ltv_bucket'] = pd.cut(
    loan_level['ltv'], bins=ltv_bins, labels=ltv_labels, right=False
)
ltv_dr = loan_level.groupby('ltv_bucket', observed=True)['default'].mean() * 100
ltv_dr.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Default Rate by LTV Bucket')
axes[1].set_ylabel('Default Rate (%)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=2))

# Top 15 states by loan count
top_states = loan_level['property_state'].value_counts().head(15).index
state_dr = (
    loan_level[loan_level['property_state'].isin(top_states)]
    .groupby('property_state')['default'].mean() * 100
    .sort_values(ascending=False)
)
state_dr.plot(kind='bar', ax=axes[2], color='seagreen', edgecolor='white')
axes[2].set_title('Default Rate by State (top 15)')
axes[2].set_ylabel('Default Rate (%)')
axes[2].tick_params(axis='x', rotation=45)
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(decimals=2))

plt.tight_layout()
plt.savefig(FIGURES / 'eda_default_rate_by_segment.png', bbox_inches='tight')
plt.show()

## 3. FICO vs. Default boxplot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = loan_level.dropna(subset=['credit_score'])
plot_data['default_label'] = plot_data['default'].map({0: 'No Default', 1: 'Default'})
sns.boxplot(
    data=plot_data, x='default_label', y='credit_score',
    palette={'No Default': 'steelblue', 'Default': 'crimson'}, ax=ax
)
ax.set_title('FICO Score vs. Default Outcome')
ax.set_xlabel('')
ax.set_ylabel('FICO Score')
plt.tight_layout()
plt.savefig(FIGURES / 'eda_fico_vs_default_boxplot.png', bbox_inches='tight')
plt.show()

print(f"Median FICO | No Default: {plot_data[plot_data['default']==0]['credit_score'].median():.0f}")
print(f"Median FICO | Default:    {plot_data[plot_data['default']==1]['credit_score'].median():.0f}")

## 4. Delinquency status distribution

In [ ]:
delinq_map = {'0': 'Current', '1': '30 DPD', '2': '60 DPD', '3': '90 DPD'}
delinq_counts = (
    panel['delinquency_status']
    .where(panel['delinquency_status'].isin(delinq_map))
    .fillna('REO/Other')
    .map(lambda x: delinq_map.get(x, x))
    .value_counts()
)

fig, ax = plt.subplots(figsize=(8, 4))
delinq_counts.plot(kind='bar', ax=ax, color=['steelblue','gold','darkorange','crimson','purple'])
ax.set_title('Delinquency Status Distribution (2021-2025 panel, monthly observations)')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig(FIGURES / 'eda_delinquency_distribution.png', bbox_inches='tight')
plt.show()

print(delinq_counts)

## 5. Loan age distribution (right-censoring check)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(
    loan_level['max_loan_age'].clip(0, 60), bins=60,
    color='steelblue', edgecolor='none', alpha=0.8
)
ax.set_xlabel('Maximum Observed Loan Age (months)')
ax.set_ylabel('Number of Loans')
ax.set_title('Loan Age at Last Observation — Right-Censoring Check\n(spike at right = many loans still active)')
plt.tight_layout()
plt.savefig(FIGURES / 'eda_loan_age_distribution.png', bbox_inches='tight')
plt.show()

pct_young = (loan_level['max_loan_age'] < 24).mean()
print(f'{pct_young:.1%} of loans have fewer than 24 months of history (heavily censored — Cox PH handles this)')

## 6. Zero balance code breakdown

In [ ]:
zbc_map = {
    '01': 'Prepaid',
    '02': 'Third-party sale',
    '03': 'Short sale',
    '06': 'Repurchase',
    '09': 'REO/Foreclosure',
}
zbc = (
    panel[panel['zero_balance_code'].notna()]
    ['zero_balance_code']
    .map(lambda x: zbc_map.get(str(x).strip(), f'Other ({x})'))
    .value_counts()
)

fig, ax = plt.subplots(figsize=(8, 4))
zbc.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Exit Type Breakdown (rows with zero_balance_code set)')
ax.set_xlabel('Count')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig(FIGURES / 'eda_zero_balance_breakdown.png', bbox_inches='tight')
plt.show()

print(zbc)

## 7. Vintage default rate chart (by origination quarter)

In [ ]:
# Use the full CPH panel (2018-2025) for the vintage chart to show more history
panel_cph['orig_dt'] = pd.to_datetime(
    panel_cph['first_payment_date'].astype(str), format='%Y%m', errors='coerce'
)
panel_cph['orig_quarter'] = panel_cph['orig_dt'].dt.to_period('Q')
terminal_c = panel_cph['zero_balance_code'].isin(['03', '06', '09'])
dpd90_c    = pd.to_numeric(panel_cph['delinquency_status'], errors='coerce').fillna(0) >= 3
panel_cph['default'] = (terminal_c | dpd90_c).astype(int)

vintage = (
    panel_cph.groupby(['loan_id', 'orig_quarter'])['default'].max()
    .reset_index()
    .groupby('orig_quarter')['default'].mean() * 100
)

fig, ax = plt.subplots(figsize=(14, 5))
vintage.plot(ax=ax, marker='o', markersize=3, linewidth=1.5, color='steelblue')
ax.set_title('Loan-Level Default Rate by Origination Quarter (2018-2025)')
ax.set_xlabel('Origination Quarter')
ax.set_ylabel('Default Rate (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=2))
ax.tick_params(axis='x', rotation=45)

# Annotate COVID cohort
if '2020Q1' in vintage.index.astype(str):
    ax.axvspan(
        vintage.index.get_loc('2020Q1') - 0.5,
        vintage.index.get_loc('2020Q2') + 0.5,
        alpha=0.15, color='crimson', label='COVID (2020Q1-Q2)'
    )
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES / 'eda_vintage_default_rates.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics table
print('=== Summary Statistics (loan level, 2021-2025) ===')
summary = loan_level[['credit_score', 'ltv', 'dti', 'orig_loan_term']].describe().round(1)
print(summary.to_string())